<a href="https://colab.research.google.com/github/Terenceisstudying/SIT-UofG-QC-Assignment/blob/main/BB84-Plain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BB84 Quantum Key Distribution without an Attacker

This notebook simulates the BB84 quantum key distribution protocol without deliberately inserting an attacker. The aim is to show how Alice and Bob create a shared secret key from quantum states, while still following the main checks used in true BB84.

Important idea: Bob is not expected to recover all of Alice's original bits. Bob measures every qubit, but only the positions where Alice and Bob used the same basis are kept. The other positions are discarded because Bob measured in the wrong basis and the result is random.

Even though this is the plain no-attacker version, true BB84 still estimates the error rate. Noise or eavesdropping can cause Alice and Bob's sifted bits to disagree, so the protocol checks a public sample of bits before accepting the final key.

## Protocol overview

1. Alice generates random classical bits and random basis choices.
2. Alice encodes each bit as a qubit.
3. Bob independently chooses random bases and measures every qubit.
4. Bob publicly announces his bases, and Alice says which positions matched her bases.
5. Alice and Bob keep only matching-basis positions. This is called sifting.
6. Alice and Bob publicly compare a sample of the sifted bits to estimate QBER, the quantum bit error rate.
7. If QBER is too high, they abort. If QBER is acceptable, they remove the public test bits and keep the remaining bits as the final key.

Basis convention used in this notebook:

- `0` means standard basis, shown as `s`.
- `1` means diagonal basis, shown as `d`.
- `0` in standard basis is `|0>`.
- `1` in standard basis is `|1>`.
- `0` in diagonal basis is `|+>`.
- `1` in diagonal basis is `|->`.

In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math
from html import escape
from IPython.display import HTML, display

## Quantum randomness

To generate a random bit by applying a Hadamard gate to `|0>`, then measures it. The result is either `0` or `1` with equal probability.  
`|+> = (|0> + |1>) / sqrt(2)`

In [2]:
# Extra helper functions for getting random number with quantum measurement instead of random number generator.

def quantum_random_int(upper_bound):
    # Return a quantum-random integer from 0 to upper_bound - 1.
    if upper_bound <= 0:
        raise ValueError("upper_bound must be positive")

    bit_count = max(1, (upper_bound - 1).bit_length())

    # Rejection sampling avoids bias when upper_bound is not a power of 2.
    while True:
        value = 0
        for _ in range(bit_count):
            value = (value << 1) | random_quantum_bit()
        if value < upper_bound:
            return value


def quantum_random_float(precision_bits=8):
    # Approximate a random float in [0, 1) using quantum-random bits.
    value = 0
    for _ in range(precision_bits):
        value = (value << 1) | random_quantum_bit()
    return value / (2 ** precision_bits)

In [3]:
# Creates random Quantum bit.
# To generate genuinely random numbers, we need a physical process that is random.
# Applying a Hadamard gate
# For example: construct |+> and measure it.

def random_quantum_bit():
    qc = QuantumCircuit(1,1)
    qc.h(0)
    qc.measure(0,0)

    backend = BasicSimulator()
    compiled_circuit = transpile(qc, backend)
    job = backend.run(compiled_circuit, shots=1)
    result = job.result()
    counts = result.get_counts()

    return int(list(counts.keys())[0])

# List of independent qubits.
def random_quantum_bits(n):
    return [random_quantum_bit() for _ in range(n)]

## Alice's encoding and optional channel noise

Alice uses her bit and basis to prepare the correct qubit. In this `BB84-Plain.ipynb` notebook, `channel_noise_probability` is set to `0.0`, so the quantum channel does not change the qubit. The noise function is included because real BB84 still needs to check for errors even when no attacker is expected.

In [4]:
# Alice encodes each bit
# bit|basis   |state
# 0  |standard| |0>
# 1  |standard| |1>
# 0  |diagonal| |+>
# 1  |diagonal| |->
def alice_encode_qubit(bit, basis):
    qc = QuantumCircuit(1, 1)

    # A bit value of 1 starts from |1>; a bit value of 0 stays as |0>.
    if bit == 1:
        qc.x(0)

    # A diagonal-basis qubit is created by applying H after the bit is set.
    if basis == 1:
        qc.h(0)

    return qc


def apply_channel_noise(prepared_qubit, noise_probability):
    qc = prepared_qubit.copy()

    # The default plain setting is 0.0, so no noise is applied.
    if noise_probability <= 0:
        return qc

    # This simple noise model flips the qubit with a chosen probability.
    if quantum_random_float() < noise_probability:
        qc.x(0)

    return qc

## Bob's measurement

Bob measures every qubit. If Bob uses the same basis as Alice, his measured bit should match Alice's bit in the ideal no-noise case. If Bob uses the wrong basis, the result is random, so that position is not used for the final key.

In [5]:
# Bob measures qubit

def bob_measure(prepared_qubit, basis):
    qc = prepared_qubit.copy()

    # if measuring in in diagonal basis (1), apply H
    if basis == 1:
        qc.h(0)
    # if measuring in standard basis (0), just do it
    qc.measure(0, 0)

    backend = BasicSimulator()
    compiled = transpile(qc, backend)
    job = backend.run(compiled, shots=1)
    result = job.result()
    counts = result.get_counts(compiled)

    return int(list(counts.keys())[0])

## Lecture 3b table helpers

The table marks the matching-basis columns in red. Those red positions are the only positions eligible for the sifted key. This is for easy identification of the output.

In [6]:
# Converting of the basis to 's' standard, 'd' diagonal to follow the lecture's convention
def basis_label(basis):
      if basis == 0:
          return "s"
      return "d"
# Converting of the qubit label to follow the lecture's convention
def qubit_label(bit, basis):
      if basis == 0:
          return str(bit)
      if bit == 0:
          return "+"
      return "-"
# Converting of Bob's bit that does not match Alice's basis to follow the lecture's convention
def bob_display_bit(i, alice_basis, bob_basis, bob_bits):
      if alice_basis[i] == bob_basis[i]:
          return str(bob_bits[i])
      return "?"

def table_cell(value, is_red=False):
    colour = "#c00000" if is_red else "#111111"
    weight = "700" if is_red else "400"
    return f"<td style='color:{colour}; font-weight:{weight}; padding:4px 8px; text-align:center;'>{escape(str(value))}</td>"


def display_bb84_table(alice_bits, alice_basis, bob_basis, bob_bits, matching_positions):
    matching_set = set(matching_positions)
    n = len(alice_bits)

    rows = [
        ("Index", [i for i in range(n)], True),
        ("A bit", alice_bits, True),
        ("A basis", [basis_label(basis) for basis in alice_basis], True),
        ("qubit", [qubit_label(alice_bits[i], alice_basis[i]) for i in range(n)], False),
        ("B basis", [basis_label(basis) for basis in bob_basis], True),
        ("B bit", [bob_display_bit(i, alice_basis, bob_basis, bob_bits) for i in range(n)], True),
        ("B raw", bob_bits, False),
    ]

    html = """
    <style>
      .bb84-table { border-collapse: collapse; font-family: monospace; margin-top: 8px; }
      .bb84-table th { padding: 4px 10px; text-align: right; color: #111111; }
      .bb84-caption { font-family: sans-serif; margin-top: 8px; color: #333333; }
    </style>
    <table class='bb84-table'>
    """

    for label, values, colour_matching in rows:
        html += f"<tr><th>{escape(label)}:</th>"
        for i, value in enumerate(values):
            html += table_cell(value, colour_matching and i in matching_set)
        html += "</tr>"

    html += "</table>"
    html += "<div class='bb84-caption'>Red columns are positions where Alice and Bob used the same basis. These positions form the sifted key before public testing.</div>"
    display(HTML(html))

## Sifting, QBER testing, and reconciliation

After Bob measures, Alice and Bob publicly compare only their basis choices. They keep matching-basis positions as the sifted key. Then they reveal a sample of the sifted bits to estimate QBER. Revealed sample bits are removed from the final secret key because everyone now knows them.

In a real BB84 system, information reconciliation uses a proper classical error-correction protocol such as Cascade or LDPC codes. With no attacker and no noise, there should be no mismatches to correct.

The code also reports any unreconciled mismatches left after public testing. In this `BB84-Plain.ipynb` notebook, this list should be empty.

In [7]:
def choose_quantum_random_sample(items, sample_count):
    # Select sample_count items without using Python's random module.
    remaining = list(items)
    sample = []

    while remaining and len(sample) < sample_count:
        random_index = quantum_random_int(len(remaining))
        sample.append(remaining.pop(random_index))

    return sample


def estimate_qber(alice_bits, bob_bits, test_positions):
    if len(test_positions) == 0:
        return 0, 0.0

    error_count = 0
    for i in test_positions:
        if alice_bits[i] != bob_bits[i]:
            error_count += 1

    return error_count, error_count / len(test_positions)


def reconciliation_summary(final_alice_key, final_bob_key):
    mismatches = []
    for i in range(len(final_alice_key)):
        if final_alice_key[i] != final_bob_key[i]:
            mismatches.append(i)

    if len(mismatches) == 0:
        return "No reconciliation correction needed. The final keys already match.", mismatches

    return "Reconciliation would be needed here:", mismatches

## Run the plain BB84 simulation

Used `n = 50` insteaf of `n = 20` from the lecture 3b as the sample is too small.    
Can increase `n` as needed.  
The default channel noise is `0.0`, so `BB84-Plain.ipynb` notebook should normally produce QBER `0.0` and matching final keys.

In [9]:
# Constant that can be change as needed
n = 50
test_fraction = 0.25
# Quantum Bit Error Rate
qber_threshold = 0.2
channel_noise_probability = 0.0


# Alice creates random bits and random bases.
alice_bits = random_quantum_bits(n)
alice_basis = random_quantum_bits(n)
qubits = []
for i in range(n):
    prepared_qubit = alice_encode_qubit(alice_bits[i], alice_basis[i])
    # Simulate noise (here is redundant since noise is 0)
    transmitted_qubit = apply_channel_noise(prepared_qubit, channel_noise_probability)
    qubits.append(transmitted_qubit)

# Bob chooses random bases and measures every qubit.
bob_basis = random_quantum_bits(n)
bob_bits = []
for i in range(n):
    bob_bits.append(bob_measure(qubits[i], bob_basis[i]))

# Alice and Bob publicly basis comparison (note that it is not bit comparison)
matching_positions = []
for i in range(n):
    if alice_basis[i] == bob_basis[i]:
        matching_positions.append(i)

# Alice and Bob keep only the bits where their basis matched
sifted_alice_key = [alice_bits[i] for i in matching_positions]
sifted_bob_key = [bob_bits[i] for i in matching_positions]

# Choose some sifted positions for public error-rate testing.
test_count = int(len(matching_positions) * test_fraction)
if len(matching_positions) > 0:
    test_count = max(1, test_count)
test_positions = choose_quantum_random_sample(matching_positions, test_count)
test_position_set = set(test_positions)

# Publicly revealed test bits are removed from the final secret key.
secret_positions = []
for i in matching_positions:
    if i not in test_position_set:
        secret_positions.append(i)

error_count, qber = estimate_qber(alice_bits, bob_bits, test_positions)
channel_accepted = qber <= qber_threshold

final_alice_key = [alice_bits[i] for i in secret_positions] if channel_accepted else []
final_bob_key = [bob_bits[i] for i in secret_positions] if channel_accepted else []
reconciliation_message, final_mismatches = reconciliation_summary(final_alice_key, final_bob_key)

display_bb84_table(alice_bits, alice_basis, bob_basis, bob_bits, matching_positions)

print("Bob raw measured bits:", bob_bits)
print("Matching positions:", matching_positions)
print("Sifted Alice key:", sifted_alice_key)
print("Sifted Bob key:  ", sifted_bob_key)
print("Public QBER test positions:", test_positions)
print("Secret positions after removing public test bits:", secret_positions)
print("QBER errors:", error_count)
print("QBER:", qber)
print("QBER threshold:", qber_threshold)
print("Channel accepted:", channel_accepted)
print("Information reconciliation:", reconciliation_message)
print("Final mismatches after public testing:", final_mismatches)
print("Final Alice key:", final_alice_key)
print("Final Bob key:  ", final_bob_key)
print("Final keys match:", final_alice_key == final_bob_key)

Index:,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49
A bit:,1,1,1,0,1,1,0,1,1,0,1,1,0,1,1,1,1,1,0,0,1,0,1,1,1,0,0,1,1,1,1,0,1,0,0,1,0,0,1,1,1,1,1,1,1,0,0,1,0,0
A basis:,d,s,s,s,s,s,s,s,d,d,d,s,s,s,d,s,s,s,d,s,d,d,d,d,s,d,s,d,d,s,s,d,d,d,s,s,s,s,s,s,s,s,d,d,s,s,d,s,s,d
qubit:,-,1,1,0,1,1,0,1,-,+,-,1,0,1,-,1,1,1,+,0,-,+,-,-,1,+,0,-,-,1,1,+,-,+,0,1,0,0,1,1,1,1,-,-,1,0,+,1,0,+
B basis:,s,s,d,s,d,d,d,s,s,s,s,d,s,s,s,s,d,d,s,s,s,s,d,s,d,s,s,d,s,d,d,d,s,d,s,s,d,d,s,d,s,d,s,s,d,s,s,d,s,s
B bit:,?,1,?,0,?,?,?,1,?,?,?,?,0,1,?,1,?,?,?,0,?,?,1,?,?,?,0,1,?,?,?,0,?,0,0,1,?,?,1,?,1,?,?,?,?,0,?,?,0,?
B raw:,1,1,0,0,0,1,1,1,0,1,0,1,0,1,1,1,0,1,1,0,0,0,1,0,1,0,0,1,1,1,0,0,0,0,0,1,1,0,1,1,1,0,0,1,1,0,0,0,0,0


Bob raw measured bits: [1, 1, 0, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0]
Matching positions: [1, 3, 7, 12, 13, 15, 19, 22, 26, 27, 31, 33, 34, 35, 38, 40, 45, 48]
Sifted Alice key: [1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0]
Sifted Bob key:   [1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0]
Public QBER test positions: [3, 31, 12, 34]
Secret positions after removing public test bits: [1, 7, 13, 15, 19, 22, 26, 27, 33, 35, 38, 40, 45, 48]
QBER errors: 0
QBER: 0.0
QBER threshold: 0.2
Channel accepted: True
Information reconciliation: No reconciliation correction needed; the final keys already match.
Final mismatches after public testing: []
Final Alice key: [1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 0]
Final Bob key:   [1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 0]
Final keys match: True
